<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 Lesson 3 精华 — Human-in-the-Loop

**一句话定位**:**敏感动作前暂停,等用户决定** —— 用 `interrupt` + `Command(resume=...)` 把 4 种用户操作(accept / edit / respond / ignore)接入 agent,让 ambient agent 既能自动跑、又不会闯祸。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**🤔 学之前先想清楚:为什么 HITL 是 ambient agent 的「生死关卡」?**

| 场景 | Chat Agent | Ambient Agent |
|------|-----------|----------------|
| 你在不在场? | **在**,实时盯着 | **不在**,后台跑 |
| agent 出错了 | 你立刻发现,**当场纠正** | 错已铸成 → 邮件已发、会议已订、合同已签 |
| 风险承受度 | 高(出错没大事) | **极低**(每次错误都是真实世界事故) |

→ ambient agent 必须有 **「**安全带**」**。HITL 就是这条安全带:让人在 **不可撤销的关键动作前** 介入。

本节学会的话,你就能把任何 agent 改造成 **「**敢部署**」** 的形态。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧩 HITL 三件套 — 这就是全部物理基础

</div>

整个 HITL 系统的底层 **只有 3 个概念**。学会这 3 个,后面所有花活都是组合。

| 概念 | 是什么 | 来自 |
|------|--------|------|
| **`interrupt(payload)`** | 在节点内调用 → graph **暂停**,把 `payload` 传给前端 | `langgraph.types` |
| **`Command(resume=value)`** | 用户决定后,把 `value` 当作 `interrupt(...)` 的返回值 **续跑** graph | `langgraph.types` |
| **`Agent Inbox`** | 一个 **UI 包装**,本质就是把 interrupted threads 显示出来 + 发 `Command(resume=...)` 回去 | [dev.agentinbox.ai](https://dev.agentinbox.ai/) |

**关系图**:

```
节点代码:                          前端 UI:
                                   
user_response =                    │
  interrupt({"action":"..", ←──── 把 payload 渲染成按钮 / 表单
             "args":...})           │
                ⏸️ 暂停              │
                                   👤 用户点 Accept / Edit / ...
                ⏯️ 续跑              │
user_response =                    │
  {"type": "accept"}   ◀──────── Command(resume=[{...}])
```

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 关键理解:`interrupt` 是基于 checkpointer 的「**暂停信号**」**

你必须 **先编译图时挂上 checkpointer**,`interrupt` 才能工作。原理:

1. 节点跑到 `interrupt(payload)` 这行 → **抛出特殊异常**
2. LangGraph 捕获异常 → **保存当前 state** 到 checkpoint(用 thread_id 标识)
3. graph 返回控制权给调用方,**带上 payload**
4. 任何时候(几秒、几天后),用 `Command(resume=...)` + 同一个 `thread_id` 调用 → **从 checkpoint 恢复 state**,从 interrupt 那行 **继续往下跑**(此时 `resume` 的值 = `interrupt(...)` 的返回值)

**没 checkpointer ≠ 没暂停能力,而是连 暂停 这个动作都做不到**(state 没地方存)。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🚦 在哪里加 interrupt? — 2 个位点

</div>

本课的 email assistant 在 **2 个位置** 加了 interrupt,各有清晰的目的:

### ① Triage Interrupt — 分类 `notify` 后

**节点**:`triage_interrupt_handler`

**触发**:router 把邮件分类成 `notify`(不是 `respond`,不是 `ignore`)。这种邮件 = **「有点意思但不一定要回」**,需要用户决定。

**用户能选**:`ignore`(结束)/ `response`(给反馈 → 触发 response_agent 回信)。

### ② Tool Interrupt — 敏感工具调用前

**节点**:`interrupt_handler`(包在 response_agent 里)

**触发**:LLM 想调以下 **3 个敏感工具** 之一:

| 工具 | 为什么敏感 |
|------|----------|
| `write_email` | **不可撤回** — 发出去就没了 |
| `schedule_meeting` | **影响别人日历** — 改别人时间表 |
| `Question` | **要打扰用户** — 需要人介入回答 |

**用户能选**:`accept` / `edit` / `response` / `ignore`(4 选 1)。

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**💡 设计哲学:不是所有工具都需要 HITL**

`check_calendar_availability`(只是查日历,**无副作用**)就 **没** 加 interrupt —— 直接跑就行。

→ **HITL = 给「**会改变真实世界**」的工具加防线**,纯查询类工具放心让 agent 跑。这是平衡 **自动化 vs 控制** 的关键。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ✅ 4 种用户操作 — 完全对照表

</div>

| 操作 | resume payload | 系统行为 | 想象的真实场景 |
|------|----------------|---------|--------------|
| **`accept`** | `[{"type": "accept"}]` | **用原参数** 执行工具 | 「这封邮件写得挺好,**就这么发**吧」 |
| **`edit`** | `[{"type": "edit", "args": {"to":"...", "subject":"...", "content":"..."}}]` | **用新参数** 替换 tool_call 并执行 | 「改一下,duration 用 30 不是 45 分钟」 |
| **`response`** | `[{"type": "response", "args": "自然语言反馈"}]` | 把反馈作为 ToolMessage 加进 state → **LLM 重新生成** | 「想要 30 分钟,下午开,语气随意点」 |
| **`ignore`** | `[{"type": "ignore"}]` | **取消执行** → 整个 graph END | 「**别发了,我自己来**」 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 学生最容易混淆的点:`edit` vs `response`**

两者表面相似(都是「不同意原版」),但实现机制完全不同:

| 维度 | `edit` | `response` |
|------|--------|-----------|
| 用户输入 | **完整的新参数** dict | **自然语言** 反馈 |
| 谁理解 | **直接套用**,不需要 LLM 再过 | **LLM 重新跑一遍** 才能转化成新参数 |
| 适合 | 改 1-2 个具体字段(duration、subject) | 改 **风格 / 语气 / 思路**(很难精确描述的偏好) |
| 容错性 | **精确**,改啥是啥 | **取决于 LLM 理解**,有时会偏 |
| 用户成本 | 高(要写完整 args) | 低(说人话就行) |

**简记口诀**:**改值用 edit,改感觉用 response**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ interrupt_handler 节点的真正样子

</div>

看一下 `interrupt_handler` 的核心结构(简化版,去掉细节噪声):

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">HITL_TOOLS = ["write_email", "schedule_meeting", "Question"]

def interrupt_handler(state):
    last_msg = state["messages"][-1]
    new_messages = []

    for tool_call in last_msg.tool_calls:
        # 1️⃣ 不敏感的工具直接跑
        if tool_call["name"] not in HITL_TOOLS:
            observation = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
            new_messages.append(ToolMessage(observation, tool_call_id=tool_call["id"]))
            continue

        # 2️⃣ 敏感工具 → 构造 Agent Inbox 看得懂的 payload
        request = {
            "action_request": {"action": tool_call["name"], "args": tool_call["args"]},
            "config": {"allow_accept": True, "allow_edit": True,
                       "allow_respond": True, "allow_ignore": True},
            "description": f"原邮件: {state['email_input']}\n\n提议: {tool_call}"
        }

        # 3️⃣ 暂停,等用户
        response = <span style="background:#3d3414; color:#fde047; padding:0 2px;">interrupt([request])</span>[0]
        # response 形如 {"type": "accept"} / {"type": "edit", "args": {...}} / ...

        # 4️⃣ 根据用户回应,分支处理
        if response["type"] == "accept":
            observation = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
            new_messages.append(ToolMessage(observation, tool_call_id=tool_call["id"]))

        elif response["type"] == "edit":
            new_args = response["args"]
            # ⚠️ 关键:改 tool_call 本身,不只改 args!
            <span style="background:#3d3414; color:#fde047; padding:0 2px;">last_msg.tool_calls[i]["args"] = new_args</span>
            observation = tools_by_name[tool_call["name"]].invoke(new_args)
            new_messages.append(ToolMessage(observation, tool_call_id=tool_call["id"]))

        elif response["type"] == "response":
            # 把用户反馈作为 ToolMessage 加进 state,LLM 下一轮会看到
            new_messages.append(ToolMessage(
                f"User feedback: {response['args']}",
                tool_call_id=tool_call["id"]
            ))

        elif response["type"] == "ignore":
            return Command(goto=END)   # 用户说不,直接结束

    return {"messages": new_messages}
</code></pre>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 最坑的细节:`edit` 时必须同时修改 tool_call 本身,不能只用新 args 调工具**

为什么?想象一下 message 历史会变成什么样:

**❌ 错误做法**(只用新参数调工具,不改 tool_call):

```
[AIMessage tool_calls=[schedule_meeting(duration=45)]]    ← 说要订 45 分钟
[ToolMessage "Meeting scheduled for 30 min"]              ← 实际订了 30 分钟
```

**LLM 下一轮看到这对消息会困惑**:「我说 45,工具回我 30,**是工具出错了吗?**」→ 可能会试图「**修正**」,导致 agent 行为漂移。

**✅ 正确做法**(原地改 tool_call 的 args):

```
[AIMessage tool_calls=[schedule_meeting(duration=30)]]   ← 改成 30(看起来 LLM 一开始就这么说)
[ToolMessage "Meeting scheduled for 30 min"]             ← 跟工具调用一致
```

→ **message 历史保持一致**,LLM 下一轮不会困惑。

**学生记忆口诀**:「**edit 不是后修,是**改剧本**」—— 不是让工具按新参数执行,而是 **把剧本里 LLM 说的话也一起改了**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### ❓ 新工具 `Question` — HITL 解锁的新能力

</div>

本课新增了一个工具叫 `Question`。它 **看似不起眼,实则是 HITL 时代的代表性新能力**。

### 🤔 没有 Question 工具时,agent 信息不全怎么办?

**只有 2 个糟糕选择**:

| 选择 | 结果 |
|------|------|
| **A. 猜** | 邮件回得似是而非,**用户被坑** |
| **B. 放弃** | 让用户自己处理,agent 没价值 |

### 💡 有了 Question,出现第三种选择 — **主动问**

```python
class Question(BaseModel):
    """Ask the user for clarification."""
    content: str = Field(description="The question to ask")
```

调用流程:

1. LLM 觉得信息不全 → `tool_calls=[Question(content="周几方便?")]`
2. `interrupt_handler` 检测到 `Question`,**触发 interrupt**
3. 问题显示在 Agent Inbox → 用户回答
4. `response` 类型的 resume 把答案塞进 messages
5. LLM 下一轮看到答案,**继续工作**

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**📚 Question 的工具类型 — 信号型**

`Question` 跟 `Done` 一样是 **信号型工具**:

- 用 **Pydantic class**(不是 `@tool` 函数)
- **没有执行逻辑** —— 调用它的全部价值在「**LLM 调了它**」这个事件本身
- 真正的「执行」是 `interrupt_handler` 检测到这个名字,**触发 HITL 流程**

→ 工具不仅可以「**做事**」,还可以「**表达意图**」。这是 LangGraph 工具系统最优雅的设计。

📎 完整工具类型分类详见 [`0_c_⭐️_工具类型大全.ipynb`](./0_c_⭐️_工具类型大全.ipynb)

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 📮 Agent Inbox — 揭开 UI 的盒子

</div>

Agent Inbox 看起来是个酷炫的 UI,但它的 **本质极其简单** —— 一句话能讲完:

**Agent Inbox = `Command(resume=...)` 的可视化封装**

### 🎬 完整的数据流

```
1️⃣ 你的 graph 跑到 interrupt:
   - state 被 checkpointer 保存
   - graph 进入 "interrupted" 状态
   - thread 文件存在本地(或 Postgres)

2️⃣ Agent Inbox 通过 LangGraph SDK 查询:
   - 列出所有 "interrupted" 状态的 threads
   - 读出每个 thread 的 interrupt payload
   - 根据 payload 里的 action_request / config / description 字段,渲染 UI 元素

3️⃣ 用户点击按钮(比如 Accept):
   - Agent Inbox 构造 Command(resume=[{"type": "accept"}])
   - 通过 LangGraph SDK 发回 graph

4️⃣ Graph 续跑:
   - checkpointer 恢复 state
   - interrupt(...) 返回 [{"type": "accept"}]
   - 节点代码从这一行继续往下跑
```

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 实战提示:你完全可以不用 Agent Inbox**

Agent Inbox 只是个方便的预制 UI。在生产里你可以:

- 用 Slack bot 替代:**interrupt 时给用户发 DM,用户回复 → SDK 发 Command**
- 用自家的内部审核系统:**任何能调 LangGraph SDK 的东西都能当「审批入口」**
- 用邮件:**interrupt 时给用户发邮件,邮件里有「同意 / 拒绝」按钮(magic link)**

→ **HITL 的 UI 是开放的,只要底层用 `interrupt + Command` 就行**。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🎭 类比:HITL 像什么?

</div>

用 3 个类比帮你彻底理解 HITL 的设计哲学:

### 🅰️ 类比 1:程序员审 PR

| HITL 概念 | PR review 类比 |
|----------|----------------|
| agent 调敏感工具 | 开发者提交了一个 PR |
| interrupt | PR 进入「**等待 review**」状态 |
| Agent Inbox | GitHub PR 列表 |
| `accept` | 「Approve & Merge」 |
| `edit` | 「**Suggested Changes**」直接改代码 |
| `response` | 「Request Changes」+ comment「**这里再改改**」 |
| `ignore` | 「Close PR」 |
| Question 工具 | 开发者主动评论「**这个需求我不太明白,能澄清下吗?**」 |

### 🅱️ 类比 2:实习生向你汇报

好的实习生会在 **行动前** 跟你确认重大决策,而不是先做了再说。HITL 让 agent 变成 **「**好实习生**」**:

- 自动处理琐事(查日历)
- 关键决策(写邮件、订会)前问你
- 不确定时主动 Question,而不是瞎猜

### 🅲 类比 3:自动驾驶的「人工接管」

Tesla 自动驾驶 = ambient agent;**人工接管** = HITL。

- 高速直道(低风险)→ 自动
- 复杂路口 / 施工 / 异常情况(高风险)→ **方向盘亮红灯,等人接管**
- 人接管后给反馈 → 系统学习(下次知道这种情况要让人来)

**这 3 个类比共同传达一件事:HITL 不是 agent 的「**降级**」,是它能被信任、被部署的前提**。

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 5 个最容易踩的坑**

1. **❌ 没挂 checkpointer 就用 interrupt** → `interrupt` 抛出来但没人接,行为未定义。**`.compile(checkpointer=InMemorySaver())` 必加**。

2. **❌ `Command(resume="...")` 传字符串,但 graph 期待 list** → 我们的设计是 `interrupt([request])` 包装成 list,所以 resume 也要 list:`Command(resume=[{"type":"accept"}])`。**漏 `[]` 就报错**。

3. **❌ `edit` 时只改了 args,没改 tool_call 本身** → message 历史不一致,LLM 下一轮困惑(见上文「最坑细节」)。

4. **❌ 在 interrupt_handler 里把所有工具都拦截** → 包括 `check_calendar_availability` 这种纯查询工具。结果:**用户被每个小动作打扰**,体验暴跌。HITL 列表要 **严格** 限定在有副作用的工具。

5. **❌ interrupt payload 不带 `description`** → Agent Inbox 上用户只看到一行参数,**根本不知道这封邮件是要发给谁、为什么** → 没法判断 accept 还是 ignore。**description 是用户做决定的全部上下文**,要详细。

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 一句话带走

**HITL 的核心 = 把「**敏感的动作**」用 `interrupt` 暂停,接收用户的 4 种回复(`accept` / `edit` / `response` / `ignore`),让 ambient agent 既自动又安全**。

### 🎯 学完这节,你应该能回答:

1. ✅ **为什么 ambient agent 必须有 HITL?**(无人值守,错误不可撤销)
2. ✅ **interrupt 的物理基础是什么?**(checkpointer 保存 state,resume 时恢复)
3. ✅ **`edit` 和 `response` 怎么选?**(改值 vs 改感觉)
4. ✅ **edit 时为啥要改 tool_call 本身?**(保证 message 历史一致)
5. ✅ **`Question` 工具为啥是 class 不是 function?**(信号型工具,价值在调用本身)
6. ✅ **Agent Inbox 是什么?**(`Command(resume)` 的可视化封装,不是必须的)

### 🎁 这节解锁了什么后续

**HITL 反馈 = 学习信号**。但反馈一次性的、用完就丢就太亏了 —— 下一节 [`4_memory.ipynb`](./4_memory.ipynb) 告诉你怎么把反馈 **持久化**,让 assistant 越用越懂你。

📎 配套阅读:[`3_hitl.ipynb` 主课](./3_hitl.ipynb) · [`0_b_⭐️_edges_vs_Command.ipynb`](./0_b_⭐️_edges_vs_Command.ipynb) · [`0_c_⭐️_工具类型大全.ipynb`](./0_c_⭐️_工具类型大全.ipynb)

</div>